# agent-friend — Universal AI Tool Adapter + Agent Runtime

Write a Python function. Use it as a tool in any AI framework.

`@tool` decorator → export to **OpenAI**, **Claude**, **Gemini**, **MCP**, **JSON Schema**. Plus 51 built-in tools and a full agent runtime.

**Free to use** — get a free API key at [openrouter.ai](https://openrouter.ai/) (no credit card required).

| | |
|---|---|
| 📦 GitHub | [github.com/0-co/agent-friend](https://github.com/0-co/agent-friend) |
| 🧪 Tests | 2474 passing |
| 🔑 Free tier | OpenRouter — Gemini 2.0 Flash, Llama 3.3 70B |

---

## Setup
1. Get a free API key at [openrouter.ai](https://openrouter.ai/)
2. Run the **Install** cell
3. Enter your key in the **API Key** cell
4. Run any demo below

In [ ]:
# Install agent-friend (takes ~30 seconds)
!pip install "git+https://github.com/0-co/agent-friend.git[all]" -q
print("✓ agent-friend installed")

In [ ]:
import os
import getpass

# Option A: OpenRouter (FREE — no credit card, just an account)
# Get your key at https://openrouter.ai/
print("Get a free key at https://openrouter.ai/")
key = getpass.getpass("Enter your OpenRouter API key (sk-or-...): ")
if key:
    os.environ["OPENROUTER_API_KEY"] = key
    print("✓ OpenRouter key set")

# Option B: Anthropic
# key = getpass.getpass("Anthropic API key (sk-ant-...): ")
# os.environ["ANTHROPIC_API_KEY"] = key

# Option C: OpenAI
# key = getpass.getpass("OpenAI API key (sk-...): ")
# os.environ["OPENAI_API_KEY"] = key

## Universal Tool Adapter (no API key needed!)

The `@tool` decorator turns any Python function into an AI-framework-compatible tool. Export to OpenAI, Claude, Gemini, MCP — from a single definition. **This demo requires no API key.**

In [ ]:
from agent_friend import tool, Toolkit

@tool
def get_weather(city: str, unit: str = "celsius") -> dict:
    """Get current weather for a city.

    Args:
        city: The city name
        unit: Temperature unit (celsius or fahrenheit)
    """
    return {"temp": 22, "unit": unit, "city": city}

# Export to any framework — one function, every format
print("=== OpenAI Format ===")
print(get_weather.to_openai()[0])

print("\n=== Anthropic (Claude) Format ===")
print(get_weather.to_anthropic()[0])

print("\n=== Google Gemini Format ===")
print(get_weather.to_google()[0])

print("\n=== MCP Format ===")
print(get_weather.to_mcp()[0])

# The function still works normally
print("\n=== The function still works normally ===")
print(f"get_weather('London') → {get_weather('London')}")

=== OpenAI Format ===
{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get current weather for a city.', 'parameters': {'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'The city name'}, 'unit': {'type': 'string', 'description': 'Temperature unit (celsius or fahrenheit)'}}, 'required': ['city']}}}

=== Anthropic (Claude) Format ===
{'name': 'get_weather', 'description': 'Get current weather for a city.', 'input_schema': {'type': 'object', 'properties': {'city': {'type': 'string', 'description': 'The city name'}, 'unit': {'type': 'string', 'description': 'Temperature unit (celsius or fahrenheit)'}}, 'required': ['city']}}

=== Google Gemini Format ===
{'name': 'get_weather', 'description': 'Get current weather for a city.', 'parameters': {'type': 'OBJECT', 'properties': {'city': {'type': 'STRING', 'description': 'The city name'}, 'unit': {'type': 'STRING', 'description': 'Temperature unit (celsius or fahrenheit)'}}, 'required': ['city']}

In [ ]:
# Batch export with Toolkit
from agent_friend import Toolkit

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression.

    Args:
        expression: The math expression to evaluate
    """
    return str(eval(expression))

@tool
def search_docs(query: str, limit: int = 5) -> list:
    """Search documentation.

    Args:
        query: Search query
        limit: Max results to return
    """
    return [{"title": f"Result for {query}", "score": 0.95}]

# Combine multiple tools into a toolkit
kit = Toolkit([get_weather, calculate, search_docs])
print(kit)

# Export all tools at once
openai_tools = kit.to_openai()
print(f"\nAll {len(openai_tools)} tools in OpenAI format:")
for t in openai_tools:
    print(f"  - {t['function']['name']} ({t['type']})")

Toolkit(3 tools)

All 3 tools in OpenAI format:
  - get_weather (function)
  - calculate (function)
  - search_docs (function)


## Context Budget — Token Estimation

MCP tool definitions can consume 40–50K tokens per request. `token_report()` tells you exactly how many tokens your tools cost in each framework format.

In [ ]:
# How many tokens do your tools cost?
report = kit.token_report()
print(f"Tools: {report['tool_count']}")
print(f"\nTokens per format:")
for fmt, tokens in report['estimates'].items():
    bar = '█' * max(1, tokens // 5)
    print(f"  {fmt:13s} ~{tokens:>3d} tokens {bar}")
print(f"\nMost expensive: {report['most_expensive']}")
print(f"Least expensive: {report['least_expensive']}")

# Per-tool estimation
print(f"\nPer-tool estimates (OpenAI format):")
for t in [get_weather, calculate, search_docs]:
    tokens = t.token_estimate('openai')
    print(f"  {t.__name__:15s} ~{tokens} tokens")

## Demo 1 — Basic Chat

`Friend` is the main class. Give it a seed (system prompt) and start chatting.
Using OpenRouter's free Gemini 2.0 Flash model here.

In [ ]:
from agent_friend import Friend

friend = Friend(
    seed="You are a concise, helpful assistant.",
    model="google/gemini-2.0-flash-exp:free",  # free tier
)

response = friend.chat("What is an AI agent and why do they need special infrastructure?")
print(response.text)

## Demo 2 — Web Search

Add `"search"` to the tools list. The agent can search the web and incorporate results.

In [ ]:
researcher = Friend(
    seed="You are a research assistant. Be concise and cite sources.",
    model="google/gemini-2.0-flash-exp:free",
    tools=["search"],
)

response = researcher.chat("What are the most popular open-source AI agent frameworks in 2026?")
print(response.text)

## Demo 3 — Memory Across Turns

Add `"memory"` to persist information across conversations. Memory is stored as a local JSON file
and loaded automatically on each session.

In [ ]:
assistant = Friend(
    seed="You are a personal assistant. Remember important things about the user.",
    model="google/gemini-2.0-flash-exp:free",
    tools=["memory"],
)

# Store some information
assistant.chat("My name is Alice and I'm building an AI agent for customer support.")
assistant.chat("I prefer concise responses and I use Python.")

# Test recall — starts fresh conversation but memory persists
assistant.reset()  # clears conversation history but not memory store
response = assistant.chat("What do you know about me?")
print(response.text)

## Demo 4 — Code Execution

Add `"code"` to let the agent write and execute Python code in a sandbox.

In [ ]:
coder = Friend(
    seed="You are a coding assistant. When asked to compute, write and run Python code.",
    model="google/gemini-2.0-flash-exp:free",
    tools=["code"],
)

response = coder.chat(
    "Find the first 10 prime numbers greater than 100 and calculate their sum. Show your work."
)
print(response.text)

## Demo 5 — All Tools Together

Combine search, fetch, memory, code, and file access for a full-capability agent.

In [ ]:
full_agent = Friend(
    seed="""You are a capable personal agent. 
    You can search the web, fetch URLs, remember things, write and run code, and read/write files.
    Be direct and practical.""",
    model="google/gemini-2.0-flash-exp:free",
    tools=["search", "fetch", "memory", "code", "file"],
    budget_usd=0.10,  # spend limit — raises BudgetExceeded if exceeded
)

response = full_agent.chat(
    "Search for the latest Python release, then fetch https://www.python.org/downloads/ "
    "and tell me the current stable version."
)
print(response.text)

## Demo 6 — Fetch a URL

Fetch any URL and read its content. HTML is auto-converted to plain text. No API key needed.

In [ ]:
reader = Friend(
    seed="You summarize web content clearly and concisely.",
    model="google/gemini-2.0-flash-exp:free",
    tools=["fetch"],
)

# Fetch a real URL and summarize it
response = reader.chat("Fetch https://pypi.org/pypi/requests/json and tell me the latest version of requests and its release date.")
print(response.text)

## Demo 7 — CLI

Run agent-friend from the command line.

In [ ]:
# Interactive mode (requires terminal — run this locally, not in Colab)
# agent-friend -i --tools search,memory,code,file

# One-shot mode:
!agent-friend --model google/gemini-2.0-flash-exp:free "What time is it right now?"

## Demo 8 — Voice

Add `"voice"` to let your agent speak responses aloud. Uses system TTS (espeak on Linux, `say` on macOS, PowerShell on Windows) — no API key needed. For neural voices, point it at any HTTP TTS server.

Run this locally to hear the agent speak. In Colab, set `AGENT_FRIEND_TTS_URL` to an HTTP TTS server URL.

In [ ]:
import os
from agent_friend import Friend, VoiceTool

# VoiceTool: no API key needed for system TTS
# - Local: uses espeak (Linux), say (macOS), PowerShell (Windows) automatically
# - HTTP/Colab: set AGENT_FRIEND_TTS_URL to a TTS server for neural voices
tts_url = os.environ.get("AGENT_FRIEND_TTS_URL")

speaker = Friend(
    seed="You are a concise assistant. When asked to speak, use the voice tool.",
    model="google/gemini-2.0-flash-exp:free",
    tools=[VoiceTool(tts_url=tts_url)],
    budget_usd=0.02,
)

response = speaker.chat(
    "Use the voice tool to say: 'agent-friend can search, remember, run code, and speak.' "
    "Then tell me what happened."
)
print(response.text)
# Run locally to hear it — espeak (Linux), say (macOS), or PowerShell (Windows) required

## Demo 9 — RSS Feeds

Add `"rss"` to subscribe to RSS and Atom feeds. No API key required — just a URL.

`read_feed(name)` returns the latest items from a subscribed feed.

In [ ]:
from agent_friend import Friend

# RSSFeedTool: no API key needed — just a URL
# fetch_feed works directly; subscribe + read_feed for named shortcuts
rss_agent = Friend(
    seed="You are a helpful news summarizer.",
    tools=["rss"],
    api_key=OPENROUTER_API_KEY,
    model="google/gemini-2.0-flash-exp:free",
)

# Subscribe to HN and read the top 3 stories
response = rss_agent.chat(
    "Subscribe the Hacker News RSS at https://news.ycombinator.com/rss with name hn, "
    "then read the top 3 stories and give me a one-sentence summary of each."
)
print(response.text)

## Demo 10 — Scheduled Tasks

Add `"scheduler"` to schedule tasks for your agent to run on a timer or at a specific time.

`run_pending()` returns tasks that are due — combine with a cron job or `agent-friend schedule` CLI.

In [ ]:
from agent_friend import Friend, SchedulerTool
import datetime

# SchedulerTool: zero dependencies, stores in ~/.agent_friend/scheduler.json
scheduler_agent = Friend(
    seed="You manage scheduled tasks. Be concise about what you've scheduled.",
    tools=["scheduler"],
    api_key=OPENROUTER_API_KEY,
    model="google/gemini-2.0-flash-exp:free",
)

# Schedule a recurring task (runs every 1440 minutes = daily)
response = scheduler_agent.chat(
    "Schedule a task called 'daily_news' with prompt 'Search for AI agent news and summarize top 3' "
    "to run every 1440 minutes. Then list all scheduled tasks."
)
print(response.text)

# Directly inspect pending tasks
tool = SchedulerTool()
tasks = tool.list_scheduled()
print(f"\nScheduled tasks: {len(tasks)}")
for t in tasks:
    print(f"  - {t['id']}: runs every {t.get('interval_minutes')} min, next: {t['next_run']}")

## Demo 11 — SQLite Database

Add `"database"` to give your agent a real relational database. It can create tables, insert rows, run queries, and inspect schemas — all backed by a local SQLite file. Zero dependencies.

In [ ]:
from agent_friend import Friend, DatabaseTool

# DatabaseTool: no API key needed for the Python API
db = DatabaseTool()
db.create_table("tasks", "id INTEGER PRIMARY KEY, title TEXT, done INTEGER DEFAULT 0")
db.insert("tasks", {"title": "Ship v0.9", "done": 0})
db.insert("tasks", {"title": "Write tests", "done": 1})
db.insert("tasks", {"title": "Update Colab", "done": 1})

print(db.query("SELECT * FROM tasks"))
print(db.query("SELECT COUNT(*) as total FROM tasks WHERE done = 1"))

# With the full agent:
# agent = Friend(tools=["database"])
# agent.chat("Create a notes table and add three ideas")
# agent.chat("Show me all my notes")


## Demo 12 — Custom Tools with `@tool`

Register any Python function as an agent tool. Type hints become the JSON schema automatically.
The decorated function remains callable normally.


In [ ]:
from agent_friend import Friend, tool

# @tool turns any function into an agent tool
# Type hints → JSON schema. Optional params are not required.

@tool
def stock_price(ticker: str) -> str:
    """Get current stock price for a ticker symbol."""
    prices = {"AAPL": "182.50", "GOOG": "175.20", "NVDA": "875.00"}
    return f"{ticker.upper()}: ${prices.get(ticker.upper(), '???')}"

@tool(name="celsius_to_fahrenheit", description="Convert Celsius to Fahrenheit")
def to_fahrenheit(celsius: float) -> str:
    return f"{celsius:.1f}°C = {celsius * 9/5 + 32:.1f}°F"

# Functions still work normally
print(stock_price("AAPL"))    # AAPL: $182.50
print(to_fahrenheit(22.0))    # 22.0°C = 71.6°F

# Mix custom tools with built-in tools
agent = Friend(
    seed="You are a helpful assistant.",
    tools=["search", stock_price, to_fahrenheit],
)
response = agent.chat(
    "What's AAPL stock price? Also convert 25°C to Fahrenheit."
)
print(response.text)


## Demo 13 — Git Integration

Add `"git"` to let your agent read git status, view diffs, inspect commit history, stage files, and commit changes. Works on any git repository.


In [ ]:
from agent_friend import Friend, GitTool
import tempfile, subprocess, os

# Create a temp git repo for the demo
with tempfile.TemporaryDirectory() as repo_dir:
    subprocess.run(['git', 'init', repo_dir], capture_output=True)
    subprocess.run(['git', 'config', 'user.email', 'demo@example.com'], cwd=repo_dir, capture_output=True)
    subprocess.run(['git', 'config', 'user.name', 'Demo'], cwd=repo_dir, capture_output=True)
    open(os.path.join(repo_dir, 'main.py'), 'w').write('print("hello")')
    subprocess.run(['git', 'add', '.'], cwd=repo_dir, capture_output=True)
    subprocess.run(['git', 'commit', '-m', 'Initial commit'], cwd=repo_dir, capture_output=True)

    # Python API — no LLM needed
    git = GitTool(repo_dir=repo_dir)
    print('Status:', git.status()[:100])
    print('Log:', git.log())

    # Modify a file and commit via the agent
    open(os.path.join(repo_dir, 'main.py'), 'w').write('print("hello, world")')
    git.add(['main.py'])
    result = git.commit('Update greeting')
    print('Commit:', result[:80])
    print('New log:', git.log())

# With the full agent:
# friend = Friend(tools=['git', 'file', 'code'], api_key='...')
# friend.chat('Show me the git status and last 3 commits')
# friend.chat('Stage all changes and commit with a descriptive message')


## Demo 14 — CSV/Tabular Data

Add `"table"` to let your agent read, filter, and aggregate CSV files — no pandas required. Auto-detects comma vs tab delimiters. Supports 8 filter operators and 6 aggregation functions.


In [ ]:
from agent_friend import Friend, TableTool
import csv, io, os

# Create a sample CSV file for the demo
sample_csv = """region,product,revenue,units
North,Widget A,1200,15
South,Widget B,850,10
North,Widget B,2100,25
East,Widget A,975,12
South,Widget A,1500,18
East,Widget B,620,8
"""
csv_path = "/tmp/sales.csv"
with open(csv_path, "w") as f:
    f.write(sample_csv)

# Python API — no LLM needed
table = TableTool()
rows = table.read(csv_path)
print(f"Rows: {len(rows)}, Columns: {table.columns(csv_path)}")

# Filter: North region only
north = table.filter_rows(csv_path, "region", "eq", "North")
print(f"North rows: {north}")

# Aggregate: total revenue
total = table.aggregate(csv_path, "revenue", "sum")
avg_rev = table.aggregate(csv_path, "revenue", "avg")
print(f"Total revenue: {total}, Avg: {avg_rev}")

# Filter: high-value rows (revenue > 1000)
high_value = table.filter_rows(csv_path, "revenue", "gt", "1000")
print(f"High-value rows: {len(high_value)}")

# With the full agent:
# agent = Friend(tools=['table', 'code'], api_key='...')
# agent.chat('Read sales.csv and tell me the average revenue by region')
# agent.chat('Which products had revenue over 1000? How many units total?')


## Demo 15 — Receive Webhooks

Add `"webhook"` to let your agent *receive* incoming HTTP requests — not just make them. Payment callbacks, GitHub webhooks, form submissions, any external trigger.

This makes agents reactive, not just procedural.


In [ ]:
from agent_friend import WebhookTool
import threading, urllib.request, json as _json, time

# Simulate receiving a webhook
hook = WebhookTool(port=0)  # port=0 = auto-assign

# Send a test webhook after 0.2s
def send_test_webhook():
    time.sleep(0.2)
    # Wait for port to be assigned
    for _ in range(20):
        port = hook.get_port()
        if port:
            break
        time.sleep(0.05)
    if not port:
        return
    payload = _json.dumps({"event": "payment.success", "amount": 99.99, "currency": "USD"}).encode()
    req = urllib.request.Request(
        f"http://localhost:{port}/payment",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    try:
        urllib.request.urlopen(req, timeout=5)
    except:
        pass

threading.Thread(target=send_test_webhook, daemon=True).start()

# Listen for webhook
print("Waiting for webhook at /payment...")
result = hook.listen(path="/payment", timeout=5.0)

print(f"Received webhook!")
print(f"  Path: {result['path']}")
print(f"  JSON body: {result['json']}")
print(f"  Event: {result['json']['event']}")
print(f"  Amount: ${result['json']['amount']}")

# With the full agent:
# from agent_friend import Friend, WebhookTool
# hook = WebhookTool(port=8765)
# agent = Friend(tools=['memory', 'code', hook], api_key='...')
# agent.chat('Wait for a /payment webhook. When it arrives, log the amount to memory.')
# # curl -X POST http://localhost:8765/payment -d '{"amount": 99.99}'


## Demo 16 — HTTP REST Client

Add `"http"` to let your agent call any REST API. Unlike `FetchTool` (read-only GET), `HTTPTool` supports all methods with custom headers, JSON bodies, and auth — ideal for writing to APIs.

`default_headers` bakes in auth headers so you don't have to pass them on every call.

In [ ]:
import json as _json
import urllib.request
from agent_friend import HTTPTool

# HTTPTool: generic REST client — GET/POST/PUT/PATCH/DELETE
# stdlib only — no requests library needed

# Demo with GitHub's public API (no auth needed)
http = HTTPTool()

# GET request — fetch public JSON
result_json = _json.loads(http.execute("http_request", {
    "method": "GET",
    "url": "https://pypi.org/pypi/requests/json",
}))
latest = result_json["json"]["info"]["version"]
print(f"Latest requests version: {latest}")

# POST example (using httpbin.org for testing)
post_result = _json.loads(http.execute("http_request", {
    "method": "POST",
    "url": "https://httpbin.org/post",
    "body": {"name": "agent-friend", "version": "0.13.0"},
}))
print(f"POST echo: {post_result.get('json', {})}")

# With auth headers baked in (e.g. for a private API):
# api = HTTPTool(default_headers={"Authorization": "Bearer sk-..."})
# result = api.execute("http_request", {"method": "GET", "url": "https://api.stripe.com/v1/charges"})

# With the full agent:
# friend = Friend(tools=["search", "http", "memory"], api_key="...")
# friend.chat("GET https://api.github.com/repos/0-co/agent-friend and save the star count to memory")
# friend.chat("POST to https://api.example.com/orders with body {item: widget, qty: 5}")

## Demo 17 — Key-Value Cache with TTL

Add `"cache"` to prevent redundant API calls. Cache any value with an optional TTL — persisted to disk so it survives process restarts.

Typical use: fetch the GitHub stars once, cache for an hour. No more hitting the API on every run.

In [ ]:
import json as _json
from agent_friend import CacheTool

# CacheTool: key-value store with TTL, persisted to JSON on disk
cache = CacheTool()

# Store a value for 1 hour
cache.cache_set('gh_stars', '42', ttl_seconds=3600)
print('Cached:', cache.cache_get('gh_stars'))    # '42'

# No-expiry cache
cache.cache_set('user_pref', 'concise', ttl_seconds=None)
print('Pref:', cache.cache_get('user_pref'))     # 'concise'

# Cache stats
stats = _json.loads(cache.cache_stats())
print(f'Entries: {stats["entries"]}, Hits: {stats["session_hits"]}')

# Expired entries return None automatically
cache.cache_set('temp', 'quick', ttl_seconds=0)
import time; time.sleep(0.1)
print('Expired:', cache.cache_get('temp'))        # None

cache.cache_clear()
print('After clear:', _json.loads(cache.cache_stats())['entries'])  # 0

# With the full agent:
# from agent_friend import Friend
# friend = Friend(tools=['http', 'cache'], api_key='...')
# friend.chat(
#     'Fetch the GitHub stars for 0-co/agent-friend. '
#     'Cache the result under gh_stars for 1 hour. '
#     'If already cached, use the cached value.'
# )


## Demo 18 — Notifications

Add `"notify"` to let your agent alert you when tasks complete. Works via desktop popup
(notify-send on Linux, osascript on macOS) or falls back to a JSONL log file — so it
works in any environment, including headless servers.


In [ ]:
from agent_friend import NotifyTool
import json as _json

# NotifyTool: notify(title, message) auto-detects best channel
# - Desktop: notify-send (Linux) or osascript (macOS)
# - Fallback: JSONL log file at ~/.agent_friend/notifications.log
notifier = NotifyTool()

# Send a notification (desktop if available, file always)
result = notifier.notify('Task complete', 'Daily news summary finished.')
print(result)

# Log to file explicitly (always works, even headless)
notifier.notify_file('Report ready', 'Processed 142 items', path='/tmp/agent_notifs.log')

# Ring the terminal bell
notifier.bell()

# Read recent notifications
recent = _json.loads(notifier.read_notifications(n=5))
print(f'Recent notifications: {len(recent)}')
if recent:
    print(f'  Last: [{recent[-1]["timestamp"]}] {recent[-1]["title"]} — {recent[-1]["message"]}')

# With the full agent:
# from agent_friend import Friend
# friend = Friend(
#     seed='Run the task, then notify the user when complete.',
#     tools=['scheduler', 'notify'],
#     api_key='...'
# )
# friend.chat('Process the CSV file and notify me when done')


## Demo 19 — JSON Querying

Add `"json"` to let your agent query and transform JSON responses. Pairs perfectly with `HTTPTool` — fetch an API response, then extract exactly what you need.

Dot-notation paths, array indexing, wildcards, filter, merge — stdlib only.


In [ ]:
from agent_friend import JSONTool

jt = JSONTool()

# Example: GitHub API-style JSON response
gh_data = '{"id": 1, "full_name": "0-co/agent-friend", "stargazers_count": 42, "topics": ["ai", "python", "agents"]}'

# Extract fields by dot-notation path
print('Name:', jt.json_get(gh_data, 'full_name'))             # '"0-co/agent-friend"'
print('Stars:', jt.json_get(gh_data, 'stargazers_count'))    # '42'
print('First topic:', jt.json_get(gh_data, 'topics[0]'))    # '"ai"'

# Get all topics with wildcard
users = '[{"name": "Alice", "role": "admin"}, {"name": "Bob", "role": "user"}]'
all_names = jt.json_get(users, '[*].name')
print('All names:', all_names)                                # '["Alice", "Bob"]'

# Filter array
admins = jt.json_filter(users, 'role', '"admin"')
print('Admins:', admins)                                       # '[{"name": "Alice", ...}]'

# Set a value
updated = jt.json_set(gh_data, 'stargazers_count', '100')
print('Updated stars:', jt.json_get(updated, 'stargazers_count'))

# Merge two objects
base = '{"model": "gpt-4o", "temperature": 0.7}'
overrides = '{"temperature": 0.3, "max_tokens": 500}'
merged = jt.json_merge(base, overrides)
print('Merged config:', merged)

# With the full agent:
# from agent_friend import Friend
# friend = Friend(tools=['http', 'json', 'memory'], api_key='...')
# friend.chat('Fetch https://api.github.com/repos/0-co/agent-friend and save the star count to memory')


## Demo 20: DateTimeTool — Date and Time Operations

Get the current time, parse dates, compute differences, and convert timezones.

In [ ]:
from agent_friend import DateTimeTool

dt = DateTimeTool()

# Current time in different timezones
print("UTC:", dt.now("UTC"))
print("Tokyo:", dt.now("Asia/Tokyo"))
print("New York:", dt.now("America/New_York"))

# Parse natural language dates
print("\nParsed:", dt.parse("March 12, 2026"))
print("Parsed ISO:", dt.parse("2026-03-12T15:30:00"))

# Date arithmetic
print("\nOne week from now:", dt.add_duration(dt.now(), days=7))
print("ProductHunt in days:", dt.diff(dt.now(), "2026-03-17T08:00:00", unit="days"))

# Timezone conversion
print("\nNoon UTC → Tokyo:", dt.convert_timezone("2026-03-12T12:00:00", to_tz="Asia/Tokyo"))

# Unix timestamps
ts = dt.to_timestamp("2026-03-12T00:00:00+00:00")
print("\nTimestamp:", ts)
print("Back to ISO:", dt.from_timestamp(ts))

## Demo 21 — Run Shell Commands

Add `"process"` to let your agent run shell commands and scripts. Captures stdout, stderr, and exit codes with configurable timeouts.

An agent that can shell out to any installed tool — git, pip, curl, compilers, data-processing CLIs — without you writing subprocess boilerplate.

In [ ]:
import json as _json
from agent_friend import ProcessTool

proc = ProcessTool(timeout=15)

# Run a single command — returns stdout, stderr, returncode
result = _json.loads(proc.run("python3 --version"))
print(f"Python version: {result['stdout'].strip() or result['stderr'].strip()}")
print(f"Success: {result['success']}, exit code: {result['returncode']}")

# Check if a tool is installed
git_path = _json.loads(proc.which("git"))
print(f"\ngit found at: {git_path.get('path', 'not found')}")

ffmpeg_check = _json.loads(proc.which("ffmpeg"))
print(f"ffmpeg: {ffmpeg_check.get('path') or 'not installed'}")

# Run a multi-line script
script = """
echo "=== System Info ==="
python3 -c "import sys; print('Python:', sys.version.split()[0])"
echo "Disk free:"; df -h / | tail -1 | awk '{print $4 " free"}'
"""
r = _json.loads(proc.run_script(script))
print(f"\nScript output:\n{r['stdout']}")

# Custom cwd — run in a specific directory
r2 = _json.loads(proc.run("ls -la", cwd="/tmp"))
print(f"Files in /tmp (first few lines):")
print('\n'.join(r2['stdout'].splitlines()[:5]))

# With the full agent:
# from agent_friend import Friend
# friend = Friend(
#     seed='You are a devops assistant. Run commands to diagnose issues.',
#     tools=['process', 'file', 'memory'],
#     api_key='...'
# )
# friend.chat('Check if git and python3 are installed, then show the python version')
# friend.chat('Run pip freeze and tell me the 5 largest packages installed')

## Demo 22 — Environment Variables & .env Files

Add `"env"` to let your agent read, set, and verify environment variables.
Load API keys from `.env` files. Check required vars are set before calling external services.

Sensitive variable names (KEY, TOKEN, SECRET, PASSWORD, etc.) are automatically hidden
from `env_get` and `env_list` — so the tool is safe to give to agents running in any context.

In [ ]:
import json as _json
import os
from agent_friend import EnvTool

env = EnvTool()

# Check which required env vars are set (safe to check even sensitive names)
result = _json.loads(env.env_check(["OPENROUTER_API_KEY", "HOME", "USER", "DATABASE_URL"]))
print("Required vars check:")
print(f"  Present: {result['present']}")
print(f"  Missing: {result['missing']}")
print(f"  All OK: {result['ok']}")

# Read non-sensitive variables
print(f"\nHOME = {env.env_get('HOME')}")
print(f"USER = {env.env_get('USER', default='unknown')}")
print(f"NONEXISTENT = {env.env_get('NONEXISTENT')}")

# Sensitive variables are hidden automatically
hidden = env.env_get("SOME_API_KEY")
print(f"\nSENSITIVE var: {hidden}")

# List visible PATH-like vars
path_vars = _json.loads(env.env_list(prefix="PATH"))
print(f"\nPATH vars: {list(path_vars.keys())}")

# Set a variable in the current process
env.env_set("LOG_LEVEL", "debug")
print(f"\nLOG_LEVEL = {os.environ.get('LOG_LEVEL')}")

# Load a .env file (skip if already set, report what changed)
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.env', delete=False) as f:
    f.write("MY_APP_DB=postgres://localhost/dev\n")
    f.write("MY_APP_DEBUG=true\n")
    f.write("MY_APP_PORT=8080\n")
    dotenv_path = f.name

load_result = _json.loads(env.env_load(dotenv_path))
print(f"\n.env load result: {load_result}")
print(f"MY_APP_DB = {os.environ.get('MY_APP_DB')}")

# Verify after loading
check2 = _json.loads(env.env_check(["MY_APP_DB", "MY_APP_PORT", "MY_APP_DEBUG"]))
print(f"After .env load — all present: {check2['ok']}")

os.unlink(dotenv_path)

# With the full agent:
# from agent_friend import Friend
# friend = Friend(
#     seed='Check that required API keys are set, then proceed with the task.',
#     tools=['env', 'http', 'cache'],
#     api_key='...'
# )
# friend.chat('Load .env, check OPENAI_API_KEY is set, then fetch the GitHub stars for 0-co/agent-friend')

## Demo 23 — Cryptographic Utilities

Add `"crypto"` to let your agent generate secure tokens, hash data, sign and verify HMAC signatures, generate UUIDs, and encode/decode base64.

All operations use Python's stdlib — zero dependencies. Useful for webhook signature verification, secure token generation, and data integrity checks.


In [ ]:
import json as _json
from agent_friend import CryptoTool

crypto = CryptoTool()

# Generate a secure random token
token = _json.loads(crypto.execute('generate_token', {}))
print(f'Secure token (32 bytes): {token["token"]}')

# Hash data
digest = _json.loads(crypto.execute('hash_data', {'data': 'hello world'}))
print(f'SHA-256("hello world"): {digest["digest"]}')

# Sign a webhook payload with HMAC
payload = '{"event": "payment.completed", "amount": 99.99}'
secret = 'my-webhook-secret'
signed = _json.loads(crypto.execute('hmac_sign', {'data': payload, 'secret': secret}))
print(f'HMAC-SHA256 signature: {signed["signature"]}')

# Verify the signature
verified = _json.loads(crypto.execute('hmac_verify', {
    'data': payload,
    'secret': secret,
    'signature': signed['signature']
}))
print(f'Signature valid: {verified["valid"]}')

# Tampered payload — verification fails
tampered = _json.loads(crypto.execute('hmac_verify', {
    'data': '{"event": "payment.completed", "amount": 0.01}',
    'secret': secret,
    'signature': signed['signature']
}))
print(f'Tampered payload valid: {tampered["valid"]}')

# Generate a UUID
uid = _json.loads(crypto.execute('uuid4', {}))
print(f'UUID4: {uid["uuid"]}')

# Base64 encode/decode
encoded = _json.loads(crypto.execute('base64_encode', {'data': 'agent-friend rocks'}))
decoded = _json.loads(crypto.execute('base64_decode', {'data': encoded['encoded']}))
print(f'Base64: {encoded["encoded"]} -> {decoded["decoded"]}')

# With the full agent:
# from agent_friend import Friend
# friend = Friend(
#     seed='You handle webhooks. Verify HMAC signatures before processing.',
#     tools=['crypto', 'webhook', 'http'],
#     api_key='...'
# )
# friend.chat('Verify this GitHub webhook signature and process the push event')


## Demo 24 — Input Validation

Add `"validator"` to validate user inputs before acting on them: email addresses, URLs, IP addresses, UUIDs, JSON payloads, numeric ranges, and regex patterns.

All stdlib — zero dependencies. Returns structured results so your agent can explain what's wrong.


In [ ]:
import json as _json
from agent_friend import ValidatorTool

v = ValidatorTool()

# Email validation
print('Email validation:')
print(v.validate_email('user@example.com'))        # valid
print(v.validate_email('notanemail'))              # invalid
print(v.validate_email('user..name@example.com'))  # consecutive dots

# URL validation
print('\nURL validation:')
print(v.validate_url('https://github.com/0-co/agent-friend'))  # valid
print(v.validate_url('ftp://files.example.com'))               # ftp not allowed by default
print(v.validate_url('not-a-url'))                              # invalid

# IP address
print('\nIP validation:')
print(v.validate_ip('8.8.8.8'))         # Google DNS, global
print(v.validate_ip('192.168.1.1'))      # private
print(v.validate_ip('::1'))              # IPv6 loopback

# JSON with required keys
print('\nJSON validation:')
r = v.validate_json('{"event": "payment", "amount": 99}', required_keys=['event', 'user_id'])
print(r)  # valid=False, missing user_id

# Range check
print('\nRange validation:')
print(v.validate_range(42, min_val=0, max_val=100))    # valid
print(v.validate_range(150, max_val=100))              # too high

# Pattern matching
print('\nPattern validation:')
print(v.validate_pattern('2026-03-12', r'(\d{4})-(\d{2})-(\d{2})'))  # captures year/month/day

# With the full agent:
# from agent_friend import Friend
# friend = Friend(
#     seed='Validate all user inputs before processing. Explain any validation errors clearly.',
#     tools=['validator', 'http', 'database'],
#     api_key='...'
# )
# friend.chat('Process this signup: email=bad-email, age=200, url=not-a-url')


## Demo 25 — Metrics & Observability

Add `"metrics"` to track counters, gauges, and timers across your agent session.
Export as JSON or Prometheus text format when you need to see what your agent has been doing.

In [ ]:
import json as _json
import time
from agent_friend import MetricsTool

m = MetricsTool()

# Counters — increment by any value
m.metric_increment('api_calls')
m.metric_increment('api_calls', 3)
m.metric_increment('errors')
print('Counter:', _json.dumps(m.metric_get('api_calls'), indent=2))

# Gauges — set current value
m.metric_gauge('queue_depth', 42)
m.metric_gauge('active_sessions', 7)
print('Gauge:', _json.dumps(m.metric_get('queue_depth'), indent=2))

# Timers — measure elapsed time
timer_id = m.metric_timer_start('search_duration')
time.sleep(0.01)  # simulate work
result = m.metric_timer_stop(timer_id)
print(f'Timer: elapsed={result["elapsed_ms"]:.1f}ms')

# Summary — see everything at once
print('\nAll metrics:', _json.dumps(m.metric_list(), indent=2))

# Export as Prometheus text format
print('\nPrometheus export:')
print(m.metric_export('prometheus'))

## Demo 26 — Parameterized Templates

Add `"template"` to give your agent reusable prompt templates with `${variable}` substitution.
Save named templates, validate variables before rendering, and manage a session template library.

In [ ]:
import json as _json
from agent_friend import TemplateTool

t = TemplateTool()

# Render a one-off template
result = t.template_render(
    'Search for ${topic} news from ${start} to ${end}.',
    {'topic': 'AI agents', 'start': '2025-01-01', 'end': '2026-03-12'}
)
print('Rendered:', result['rendered'])

# Save a named template for reuse
t.template_save('email_draft',
    'Subject: ${subject}\n\nHi ${name},\n\n${body}\n\nBest,\n${sender}')
email = t.template_render_named('email_draft', {
    'subject': 'agent-friend update',
    'name': 'Developer',
    'body': 'v0.23 ships with TemplateTool.',
    'sender': 'agent'
})
print('Email:', email['rendered'])

# Validate before rendering
validation = t.template_validate(
    'Hello ${name}, your code ${code} expires in ${hours}h',
    {'name': 'Alice', 'code': '9F3K'}  # missing 'hours'
)
print('Valid:', validation['valid'], '| Missing:', validation['missing'])

# List all saved templates
print('Templates:', _json.dumps(t.template_list(), indent=2))

## Demo 27 — Text & File Diffs

Add `"diff"` to give your agent unified diffs, word-level comparison, and similarity scoring.
Uses Python's stdlib `difflib` — zero dependencies.

In [ ]:
import json as _json
from agent_friend import DiffTool

d = DiffTool()

# Unified diff between two strings
old_code = 'def greet(name):\n    return f"Hello, {name}!"\n'
new_code = 'def greet(name: str) -> str:\n    return f"Hi, {name}!"\n'
result = d.diff_text(old_code, new_code, label_a='old.py', label_b='new.py')
print('Unified diff:')
print(result['unified'])
print(f'Added: {result["added_lines"]} | Removed: {result["removed_lines"]}\n')

# Word-level diff
words = d.diff_words('the quick brown fox', 'the fast brown cat')
print('Word diff:', words['inline'])
print(f'Added: {words["added_words"]} | Removed: {words["removed_words"]}\n')

# Similarity statistics
stats = d.diff_stats('agent-friend v0.23', 'agent-friend v0.24')
print(f'Similarity: {stats["similarity"]:.0%}\n')

# Fuzzy matching from a list
matches = d.diff_similar('agnet-friend', ['agent-friend', 'django-friend', 'agentsmith'])
print('Closest matches:')
for m in matches[:3]:
    print(f'  {m["text"]}: {m["score"]:.2f}')

## Demo 28 — Retry with Circuit Breaker

In [ ]:
import json as _json
from agent_friend import RetryTool

r = RetryTool(default_delay_seconds=0.0)  # zero delay for demo

# --- retry_http: succeeds on first attempt ---
result = _json.loads(r.retry_http('GET', 'https://httpbin.org/status/200',
                                   max_attempts=3, delay_seconds=0.5))
print('retry_http success:', result.get('ok'), 'attempts:', result.get('attempts'))

# --- retry_shell: simple command ---
shell_result = _json.loads(r.retry_shell('echo "agent-friend v0.25"', max_attempts=3))
print('retry_shell:', shell_result['stdout'].strip())

# --- circuit breaker ---
r.circuit_create('demo_api', max_failures=2, reset_timeout_seconds=30)

# Successful call keeps circuit closed
resp = _json.loads(r.circuit_call('demo_api', 'GET', 'https://httpbin.org/status/200'))
print('circuit call ok:', resp.get('ok'), '| state:', resp.get('circuit_state'))

# Stats
stats = _json.loads(r.retry_status())
print('stats:', stats)

## Demo 29 — HTML Parsing

In [ ]:
import json as _json
from agent_friend import HTMLTool

h = HTMLTool()

html = """
<html><head><title>Agent News</title></head><body>
  <h1>Top Stories</h1>
  <p>Exponential back-off for all agents. <a href='/retry'>Read more</a>.</p>
  <table>
    <tr><th>Tool</th><th>Version</th></tr>
    <tr><td>RetryTool</td><td>v0.25</td></tr>
    <tr><td>HTMLTool</td><td>v0.26</td></tr>
  </table>
</body></html>
"""

print('TEXT:',   h.html_text(html)[:100])
print('LINKS:',  _json.dumps(h.html_links(html, base_url='https://example.com')))
print('HEADS:',  _json.dumps(h.html_headings(html)))
print('TABLE:',  _json.dumps(h.html_tables(html)[0]))
print('TITLE:',  h.html_meta(html)['title'])

## Demo 30 — XML Parsing

In [ ]:
import json as _json
from agent_friend import XMLTool

x = XMLTool()
xml = """<catalog>
  <book id='1' lang='en'><title>Agent Patterns</title><price>29.99</price></book>
  <book id='2' lang='fr'><title>Async Python</title><price>24.99</price></book>
</catalog>"""

print('TITLES:',  x.xml_extract(xml, 'title'))
print('ATTRS:',   x.xml_attrs(xml, 'book'))
print('FIND:',    x.xml_find(xml, ".//book[@id='2']"))
print('TAGS:',    x.xml_tags(xml))
print('VALID:',   x.xml_validate(xml))
print('DICT:', _json.loads(x.xml_to_dict(xml)))

## Demo 31 — Regex Operations

In [ ]:
import json as _json
from agent_friend import RegexTool

rx = RegexTool()

# Find all version numbers in text
text = "Released v0.26.0, then v0.27.0, then v0.28.0"
matches = _json.loads(rx.regex_findall(r"\d+\.\d+\.\d+", text))
print("Version numbers found:", matches)

# Named groups — parse an email address
result = _json.loads(rx.regex_search(
    r"(?P<user>\w+)@(?P<domain>[\w.]+)",
    "Contact alice@example.com for details"
))
print("Named groups:", result["named_groups"])

# Replace with backreference — swap first and last name
swapped = rx.regex_replace(r"(\w+)\s+(\w+)", r"\2 \1", "Alice Smith")
print("Swapped:", swapped)

# Case-insensitive findall
log = "ERROR: timeout\nwarning: slow\nERROR: retry"
errors = _json.loads(rx.regex_findall("error", log, flags=["IGNORECASE"]))
print("Errors found:", len(errors))

# Validate a pattern
print("Valid pattern:", _json.loads(rx.regex_validate(r"\d{4}-\d{2}-\d{2}"))["valid"])
print("Invalid pattern:", _json.loads(rx.regex_validate(r"(unclosed"))["valid"])

# Escape user input for literal matching
escaped = rx.regex_escape("$1.00 (special)")
found = _json.loads(rx.regex_findall(escaped, "Price: $1.00 (special) today"))
print("Literal match:", found)

## Demo 32 — Rate Limiting

In [ ]:
import json as _json
from agent_friend import RateLimitTool

r = RateLimitTool()

# Fixed window: 5 calls per 60 seconds
r.limiter_create("openai", max_calls=5, window_seconds=60, algorithm="fixed")

# Acquire: check + consume atomically
for i in range(6):
    result = _json.loads(r.limiter_acquire("openai"))
    print(f"Call {i+1}: allowed={result['allowed']}, remaining={result.get('remaining', 'N/A')}")

# Reset and try sliding window
r.limiter_delete("openai")
r.limiter_create("github", max_calls=3, window_seconds=60, algorithm="sliding")
r.limiter_acquire("github")
r.limiter_acquire("github")
status = _json.loads(r.limiter_status("github"))
print(f"\nSliding window — count: {status['count']}, remaining: {status['remaining']}")

# Token bucket: smooth rate with burst
r.limiter_create("llm", algorithm="token_bucket", rate_per_second=2.0, burst_capacity=5.0)
status = _json.loads(r.limiter_status("llm"))
print(f"\nToken bucket — tokens: {status['tokens']}, rate: {status['rate_per_second']}/s")

# List all limiters
print("\nActive limiters:", [l['name'] for l in _json.loads(r.limiter_list())])

## Demo 33 — Work Queues

In [ ]:
import json as _json
from agent_friend import QueueTool

q = QueueTool()

# FIFO work queue
q.queue_create("urls")
q.queue_push("urls", {"url": "https://example.com", "action": "scrape"})
q.queue_push("urls", {"url": "https://other.com", "action": "fetch"})
q.queue_push("urls", {"url": "https://third.com", "action": "index"})

print("FIFO queue size:", _json.loads(q.queue_size("urls"))["size"])

# Process items in order
while True:
    result = _json.loads(q.queue_pop("urls"))
    if result.get("empty"):
        break
    print(f"Processing: {result['item']['url']} ({result['item']['action']}), {result['size']} remaining")

# Priority queue — urgent tasks first
q.queue_create("alerts", kind="priority")
q.queue_push("alerts", "log rotate", priority=10)
q.queue_push("alerts", "disk full", priority=1)
q.queue_push("alerts", "CPU spike", priority=3)

print("\nProcessing alerts by priority:")
for _ in range(3):
    result = _json.loads(q.queue_pop("alerts"))
    print(f"  → {result['item']}")

# List all queues
print("\nQueues:", _json.loads(q.queue_list()))

## Demo 34 — Event Bus

In [ ]:
import json as _json
from agent_friend import EventBusTool

bus = EventBusTool()

# Subscribe to a topic
bus.bus_subscribe("new_url", "scraper")
bus.bus_subscribe("new_url", "logger")
bus.bus_subscribe("*", "auditor")  # receives ALL events

# Publish events
bus.bus_publish("new_url", {"url": "https://example.com", "priority": 1})
bus.bus_publish("new_url", {"url": "https://other.com", "priority": 2})
bus.bus_publish("alerts", {"type": "disk_full", "pct": 95})

# Check who was notified
result = _json.loads(bus.bus_publish("new_url", {"url": "https://third.com"}))
print("Notified:", result["notified"])

# Read event history
history = _json.loads(bus.bus_history("new_url", n=3))
print(f"\nHistory ({len(history)} events):")
for event in history:
    print(f"  [{event['event_id']}] {event['data']['url']}")

# Observability
stats = _json.loads(bus.bus_stats())
print(f"\nTotal events: {stats['total_events']}")
print(f"Call counts: {stats['subscriber_counts']}")

# List topics
topics = _json.loads(bus.bus_topics())
for t in topics:
    print(f"Topic '{t['topic']}': {t['event_count']} events, {len(t['subscribers'])} subscribers")

## Demo 35 — Finite State Machine

Define states and allowed transitions. Only permitted moves execute.

In [ ]:
from agent_friend.tools.state_machine import StateMachineTool
import json

sm = StateMachineTool()

# Define an order workflow
sm.sm_create("order", initial="pending",
             states=["pending", "paid", "shipped", "delivered", "cancelled"])
sm.sm_add_transition("order", "pending", "paid")
sm.sm_add_transition("order", "pending", "cancelled")
sm.sm_add_transition("order", "paid", "shipped")
sm.sm_add_transition("order", "shipped", "delivered")

# Try an invalid transition
bad = json.loads(sm.sm_trigger("order", "delivered"))  # pending → delivered: not allowed
print("Invalid:", bad["error"])

# Follow the valid path
sm.sm_trigger("order", "paid")
sm.sm_trigger("order", "shipped")
sm.sm_trigger("order", "delivered")

state = json.loads(sm.sm_state("order"))
print("State:", state["state"])

history = json.loads(sm.sm_history("order"))
for h in history:
    print(f'  {h["seq"]}. {h["from"]} → {h["to"]}')  

# Guard check
can = json.loads(sm.sm_can("order", "cancelled"))
print("Can cancel:", can["allowed"])  # False — delivered is terminal

## Demo 36 — Map/Filter/Reduce

Transform JSON arrays without CodeTool.

In [ ]:
from agent_friend.tools.map_reduce import MapReduceTool
import json

mr = MapReduceTool()

data = json.dumps([
    {"name": "Alice", "score": 90, "dept": "eng"},
    {"name": "Bob", "score": 75, "dept": "mkt"},
    {"name": "Charlie", "score": 90, "dept": "eng"},
    {"name": "Diana", "score": 55, "dept": "mkt"},
])

# Extract names
print("Names:", mr.mr_map(data, "name"))

# Filter high scorers
top = mr.mr_filter(data, "score", "gte", 80)
print("High scorers:", mr.mr_map(top, "name"))

# Average score
print("Avg score:", mr.mr_reduce(data, "score", "avg"))

# Group by department
groups = json.loads(mr.mr_group(data, "dept"))
for dept, members in groups.items():
    print(f"{dept}: {[m['name'] for m in members]}")

# Chain: top scorers joined by &
print("Stars:", mr.mr_reduce(top, "name", "join", separator=" & "))

## Demo 37 — Directed Graph

Model dependencies, detect cycles, find install order.

In [ ]:
from agent_friend.tools.graph import GraphTool
import json

g = GraphTool()
g.graph_create("deps")

# Model package dependencies
g.graph_add_edge("deps", "django", "sqlparse")
g.graph_add_edge("deps", "django", "asgiref")
g.graph_add_edge("deps", "myapp", "django")
g.graph_add_edge("deps", "myapp", "celery")
g.graph_add_edge("deps", "celery", "kombu")

# Install order (topological sort)
order = json.loads(g.graph_topo_sort("deps"))
print("Install order:", order)

# Transitive deps of myapp
deps = json.loads(g.graph_descendants("deps", "myapp"))
print("myapp depends on:", deps)

# Shortest path
path = json.loads(g.graph_path("deps", "myapp", "sqlparse"))
print("Path to sqlparse:", path["path"])

# No cycles?
cycle = json.loads(g.graph_has_cycle("deps"))
print("Has cycle:", cycle["has_cycle"])

## Demo 38 — Human-Readable Formatting

Bytes, durations, currency, ordinals, tables.

In [ ]:
from agent_friend.tools.format_tool import FormatTool
import json

f = FormatTool()

print(f.format_bytes(1_234_567))           # 1.2 MB
print(f.format_bytes(1024, binary=True))   # 1.0 KiB
print(f.format_duration(3_661))            # 1h 1m 1s
print(f.format_duration(90, verbose=True)) # 1 minute 30 seconds
print(f.format_number(1_234_567.89))       # 1,234,567.89
print(f.format_percent(0.8734))            # 87.3%
print(f.format_currency(1234.5, "EUR"))    # €1,234.50
print(f.format_ordinal(21))                # 21st
print(f.format_plural(3, "test"))          # 3 tests

# ASCII table
data = json.dumps([{"name": "Alice", "score": 90}, {"name": "Bob", "score": 75}])
print(f.format_table(data))

## Demo 39 — Full-Text Search Index

BM25 relevance ranking over JSON docs. No database.

In [ ]:
from agent_friend.tools.search_index import SearchIndexTool
import json

idx = SearchIndexTool()

docs = [
    {"id": 1, "title": "Python packaging guide", "body": "publish packages to PyPI"},
    {"id": 2, "title": "Agent memory patterns", "body": "persistent memory using SQLite"},
    {"id": 3, "title": "Rate limiting API calls", "body": "limit API calls"},
    {"id": 4, "title": "Python async programming", "body": "asyncio and coroutines"},
]
idx.index_add("articles", docs)

# Full-text search with BM25 ranking
results = json.loads(idx.index_search("articles", "python"))
for r in results:
    print(f"[{r['_score']:.3f}] {r['title']}")

# Top result for multi-word query
top = json.loads(idx.index_search("articles", "rate limit api", top_n=1))
print("Top result:", top[0]["title"])

# Index status
print(idx.index_status("articles"))

---

## What's next?

- **Full docs**: [github.com/0-co/agent-friend](https://github.com/0-co/agent-friend)
- **More agent-* tools**: [github.com/0-co/company](https://github.com/0-co/company) — 21 packages for reliability, observability, testing, safety, and state
- **Watch it being built**: [twitch.tv/0coceo](https://twitch.tv/0coceo) — an AI building a company, live
- **Custom tools**: use `@tool` to register any function — stock prices, internal APIs, database queries
- **File a bug or request a feature**: [github.com/0-co/agent-friend/issues](https://github.com/0-co/agent-friend/issues)

## Demo 40 — Hierarchical Configuration

Dot-notation keys, type coercion, env-var loading, required-key validation. Multiple named config stores.

In [ ]:
from agent_friend.tools.config_tool import ConfigTool
import json

cfg = ConfigTool()

# Set config values (dot-notation keys)
cfg.config_set('app', 'db.host', 'localhost')
cfg.config_set('app', 'db.port', 5432)
cfg.config_set('app', 'debug', True)

# Get with type coercion
print(cfg.config_get('app', 'db.host'))
print(cfg.config_get('app', 'db.port', as_type='int'))

# Set defaults (only where key not already present)
cfg.config_defaults('app', {'db.host': '127.0.0.1', 'timeout': 30})
print(cfg.config_get('app', 'db.host'))   # still 'localhost'
print(cfg.config_get('app', 'timeout'))    # 30

# Assert required keys
print(cfg.config_require('app', ['db.host', 'db.port']))

# List keys by prefix
print(cfg.config_list('app', prefix='db.'))

# Export all
print(cfg.config_dump('app'))


## Demo 41 — Text Chunker

Split long documents for LLM context windows. Chars, tokens, sentences, paragraphs, sliding window.

In [ ]:
from agent_friend.tools.chunker import ChunkerTool
import json

chunker = ChunkerTool()

doc = 'The quick brown fox jumps over the lazy dog. ' * 20

# Chunk by characters with overlap
chunks = json.loads(chunker.chunk_text(doc, max_chars=100, overlap=20))
print(f'{len(chunks)} chunks, first: {chunks[0]["char_count"]} chars, ~{chunks[0]["token_estimate"]} tokens')

# Chunk by sentences
chunks = json.loads(chunker.chunk_text(doc, max_chars=200, mode='sentences'))
print(f'Sentence chunks: {len(chunks)}')

# Batch a list
items = list(range(25))
batches = json.loads(chunker.chunk_list(items, size=10))
print(f'Batches: {len(batches)}, sizes: {[b["count"] for b in batches]}')

# Sliding window
windows = json.loads(chunker.chunk_sliding_window(doc[:200], window_chars=80, step_chars=40))
print(f'Windows: {len(windows)}, first start={windows[0]["start"]} end={windows[0]["end"]}')

# Stats
stats = json.loads(chunker.chunk_stats(doc))
print(f'Stats: {stats["char_count"]} chars, ~{stats["token_estimate"]} tokens, {stats["word_count"]} words')


## Demo 42 — Vector Store

In-memory vector store with cosine similarity search. No numpy. Build RAG pipelines without external services.

In [ ]:
from agent_friend.tools.vector_store import VectorStoreTool
import json

vs = VectorStoreTool()

# Store embeddings (in practice, from Anthropic/OpenAI embedding API)
vs.vector_add('docs', [0.1, 0.9, 0.3], metadata={'text': 'cats and kittens'}, doc_id='doc1')
vs.vector_add('docs', [0.8, 0.1, 0.5], metadata={'text': 'dogs and puppies'}, doc_id='doc2')
vs.vector_add('docs', [0.15, 0.85, 0.25], metadata={'text': 'feline companions'}, doc_id='doc3')

# Cosine similarity search
results = json.loads(vs.vector_search('docs', [0.1, 0.9, 0.3], top_k=2))
for r in results:
    print(f"{r['id']}: score={r['score']:.4f} — {r['metadata']['text']}")

# Stats
print(json.loads(vs.vector_stats('docs')))

# Euclidean distance
r2 = json.loads(vs.vector_search('docs', [0.1, 0.9, 0.3], top_k=1, metric='euclidean'))
print(f'Euclidean nearest: {r2[0]["id"]}')


## Demo 43 — Timer Tool

Named stopwatches, countdown timers, and shell command benchmarking.

In [ ]:
from agent_friend.tools.timer_tool import TimerTool
import json, time

t = TimerTool()

# Stopwatch
t.timer_start('search')
time.sleep(0.05)
r = json.loads(t.timer_stop('search'))
print(f"Search took {r['elapsed_ms']:.1f}ms")

# Lap timing
t.timer_start('pipeline')
time.sleep(0.01)
json.loads(t.timer_lap('pipeline'))  # stage 1
time.sleep(0.02)
r = json.loads(t.timer_stop('pipeline'))
print(f"Pipeline laps: {r['laps']}ms")

# Countdown
t.countdown_start('timeout', 60.0)
r = json.loads(t.countdown_remaining('timeout'))
print(f"Countdown remaining: {r['remaining_s']:.1f}s, expired={r['expired']}")

# Benchmark
r = json.loads(t.timer_benchmark('echo hello', runs=3))
print(f"echo: avg={r['avg_ms']:.1f}ms min={r['min_ms']:.1f}ms max={r['max_ms']:.1f}ms")


## Demo 44 — Stats Tool

Descriptive statistics without numpy. mean/median/std, histogram, correlation, outliers, normalization, moving averages.

In [ ]:
from agent_friend.tools.stats_tool import StatsTool
import json

stats = StatsTool()

data = [2, 4, 4, 4, 5, 5, 7, 9]

# Descriptive statistics
r = json.loads(stats.stats_describe(data))
print(f"mean={r['mean']} median={r['median']} std={r['std']:.3f}")
print(f"percentiles: {r['percentiles']}")

# Correlation
r = json.loads(stats.stats_correlation([1,2,3,4,5], [2,4,6,8,10]))
print(f"r={r['r']} ({r['interpretation']})")

# Outlier detection
r = json.loads(stats.stats_outliers([1, 2, 3, 4, 100], method='iqr'))
print(f"Outliers: {[o['value'] for o in r['outliers']]}")

# Moving average
r = json.loads(stats.stats_moving_average([1,2,3,4,5,6,7], window=3))
print(f"SMA(3): {r['values']}")

# Frequency
r = json.loads(stats.stats_frequency(['a','b','a','c','a','b']))
print(f"Freq: {[(f['value'], f['count']) for f in r['frequencies']]}")


## Demo 45 — Sampler Tool

Random sampling, shuffling, weighted selection, stratified sampling, and train/test splits — all reproducible with seed.

In [ ]:
from agent_friend.tools.sampler import SamplerTool
import json

sampler = SamplerTool()

items = list(range(100))

# Deterministic sample
r = json.loads(sampler.sample_list(items, n=5, seed=42))
print(f'Sample: {r["sample"]}')  # same every time with seed=42

# Weighted — 80% chance of 'a'
r = json.loads(sampler.sample_weighted(['a', 'b', 'c'], [0.8, 0.1, 0.1], n=10, seed=1))
print(f'Weighted: {r["sample"]}')

# Train/test split
r = json.loads(sampler.random_split(items, ratios=[0.8, 0.2], seed=42))
print(f'Train: {r["sizes"][0]} items, Test: {r["sizes"][1]} items')

# Stratified
groups = {'positive': list(range(100)), 'negative': list(range(100, 200))}
r = json.loads(sampler.sample_stratified(groups, n_per_group=5, seed=42))
print(f'Stratified: {r}')


## Demo 46 — WorkflowTool

Lightweight workflow/pipeline runner. Define named workflows as sequences of built-in or custom steps. Supports retries, on_error (fail/skip/default), conditional execution, and execution history.

In [ ]:
from agent_friend.tools.workflow_tool import WorkflowTool
import json

wf = WorkflowTool()

# Text cleaning pipeline
wf.workflow_define('clean', steps=[
    {'name': 'strip', 'fn': 'strip'},
    {'name': 'upper', 'fn': 'upper'},
])
r = json.loads(wf.workflow_run('clean', input='  hello world  '))
print('Clean:', r['output'])

# Custom step
wf.step_define('double', 'def step(value, ctx): return value * 2')
wf.workflow_define('math', steps=[{'fn': 'to_int'}, {'fn': 'double'}])
r = json.loads(wf.workflow_run('math', input='21'))
print('Double 21:', r['output'])

# on_error=skip
wf.workflow_define('safe', steps=[
    {'fn': 'to_int', 'on_error': 'skip'},
    {'fn': 'to_str'},
])
r = json.loads(wf.workflow_run('safe', input='not_a_number'))
print('Safe ok:', r['ok'], 'output:', r['output'])

r = json.loads(wf.builtin_fns())
print('Built-ins:', r['functions'])


## Demo 47 — AlertTool

Threshold-based alerting. Define named rules (gt/gte/lt/lte/eq/ne/between/outside/contains/not_contains/is_empty/is_truthy), evaluate incoming values, track history with severity levels and cooldown support.

In [ ]:
from agent_friend.tools.alert_tool import AlertTool
import json

alerts = AlertTool()

# Define rules
alerts.alert_define('high_cpu', condition='gt', threshold=90.0, severity='critical', metric='cpu_pct')
alerts.alert_define('low_disk', condition='lte', threshold=5.0, severity='error', metric='disk_gb')
alerts.alert_define('error_log', condition='contains', threshold='ERROR', severity='warning')

# Evaluate values
r = json.loads(alerts.alert_evaluate('high_cpu', 95.0))
print('CPU 95:', r['fired'], r['severity'])

r = json.loads(alerts.alert_evaluate('high_cpu', 70.0))
print('CPU 70:', r['fired'])

r = json.loads(alerts.alert_evaluate('error_log', '2026-03-12 ERROR: timeout'))
print('Log fired:', r['fired'])

# Between condition
alerts.alert_define('mid_range', condition='between', threshold=10, threshold_high=20)
r = json.loads(alerts.alert_evaluate('mid_range', 15))
print('Between 10-20, value=15:', r['fired'])

# History and stats
r = json.loads(alerts.alert_history())
print('Total fired:', r['count'])

r = json.loads(alerts.alert_stats())
print('Total fires:', r['total_fires'])


## Demo 48 — LockTool

Named mutex-style locking to prevent concurrent operations. Supports TTL auto-expiry, optional blocking wait, non-blocking try, force-release, and release-all-by-owner.

In [ ]:
from agent_friend.tools.lock_tool import LockTool
import json

locks = LockTool()

# Acquire a lock
r = json.loads(locks.lock_acquire('db_write', owner='worker-1', ttl_s=30))
print('Acquired:', r['acquired'], 'owner:', r['owner'])

# Non-blocking try from another worker
r = json.loads(locks.lock_try('db_write', owner='worker-2'))
print('Try:', r['acquired'], 'held_by:', r.get('held_by'))

# Status
r = json.loads(locks.lock_status('db_write'))
print('Status:', r['held'], 'remaining_s:', r['remaining_s'])

# Release
locks.lock_release('db_write', owner='worker-1')
print('After release:', json.loads(locks.lock_status('db_write'))['held'])

# Release all for an owner
locks.lock_acquire('a', owner='worker-3')
locks.lock_acquire('b', owner='worker-3')
r = json.loads(locks.lock_release_all('worker-3'))
print('Released:', r['released_count'], 'locks:', r['locks'])

# Stats
r = json.loads(locks.lock_stats())
print('Total acquisitions:', r['total_acquisitions'])


## Demo 49 — AuditTool

Structured audit log for agent observability. Record events with type, actor, resource, metadata, severity, and outcome. Filter, aggregate, and export the log.

In [ ]:
from agent_friend.tools.audit_tool import AuditTool
import json

audit = AuditTool()

# Log events
audit.audit_log('user.login',  actor='alice', resource='auth', metadata={'ip': '1.1.1.1'})
audit.audit_log('file.delete', actor='bob',   resource='doc.txt', severity='warning')
audit.audit_log('api.call',    actor='alice', resource='/v1/data')
audit.audit_log('user.login',  actor='eve',   resource='auth', severity='error', outcome='failure', metadata={'ip': '9.9.9.9'})

# Search by actor
r = json.loads(audit.audit_search(actor='alice'))
print('Alice events:', r['total'])

# Search by metadata text
r = json.loads(audit.audit_search(text='9.9.9.9'))
print('Suspicious IP actor:', r['events'][0]['actor'])

# Stats
r = json.loads(audit.audit_stats())
print('By type:', r['by_type'])
print('By outcome:', r['by_outcome'])

# Timeline
r = json.loads(audit.audit_timeline(bucket='hour'))
print('Hour buckets:', len(r['buckets']), 'events:', r['total'])

# Export
r = json.loads(audit.audit_export())
print('Export lines:', r['count'])


## Demo 50 — BatchTool

Batch processing: map/filter/reduce/partition lists with named functions. 16 built-in functions, custom functions via Python source, error handling per item.

In [ ]:
from agent_friend.tools.batch_tool import BatchTool
import json

batch = BatchTool()

# Built-in map
r = json.loads(batch.batch_map([1, 2, 3, 4, 5], fn='square'))
print('Squares:', r['results'])

# Built-in filter
r = json.loads(batch.batch_filter(['', 'hello', '', 'world'], fn='is_truthy'))
print('Truthy:', r['results'])

# Custom function
batch.fn_define('add_tax', 'def fn(item): return round(item * 1.08, 2)')
r = json.loads(batch.batch_map([10.0, 20.0, 50.0], fn='add_tax'))
print('With tax:', r['results'])

# Reduce
r = json.loads(batch.batch_reduce([1, 2, 3, 4, 5], fn='sum'))
print('Sum:', r['result'])

# Partition
r = json.loads(batch.batch_partition([1, -2, 3, -4, 0], fn='is_truthy'))
print('Passing:', r['passing'], 'Failing:', r['failing'])

# Chunk
r = json.loads(batch.batch_chunk(list(range(10)), size=3))
print('Chunks:', r['chunks'])

# Zip
r = json.loads(batch.execute('batch_zip', {'keys': ['name', 'score'], 'lists': [['alice', 'bob'], [95, 87]]}))
print('Zip:', r['results'])


## Demo 51 — TransformTool

Structured data transformation: pick/omit/rename/coerce/flatten/unflatten/merge keys, apply transformations to lists of records.

In [ ]:
from agent_friend.tools.transform_tool import TransformTool

transform = TransformTool()

# Pick specific keys
record = {"name": "Alice", "age": 30, "email": "alice@example.com", "role": "admin"}
print("Pick:", transform.transform_pick(record, ["name", "role"]))

# Rename keys
print("Rename:", transform.transform_rename(record, {"name": "full_name", "email": "contact"}))

# Coerce types
data = {"count": "42", "active": "true", "score": "3.14"}
print("Coerce:", transform.transform_coerce(data, {"count": "int", "active": "bool", "score": "float"}))

# Flatten nested dict
nested = {"user": {"name": "Alice", "address": {"city": "Portland"}}}
print("Flatten:", transform.transform_flatten(nested))

# Unflatten back
flat = {"user.name": "Alice", "user.address.city": "Portland"}
print("Unflatten:", transform.transform_unflatten(flat))

# Deep merge
base = {"db": {"host": "localhost", "port": 5432}}
override = {"db": {"port": 5433, "name": "prod"}}
print("Merge:", transform.transform_merge([base, override]))

# Map records — apply transformation pipeline to a list of dicts
records = [
    {"first_name": "Alice", "last_name": "Smith", "age": "30", "ssn": "123-45-6789"},
    {"first_name": "Bob", "last_name": "Jones", "age": "25", "ssn": "987-65-4321"},
]
print("Map records:", transform.transform_map_records(records, 
    rename={"first_name": "name"}, 
    omit=["ssn", "last_name"],
    coerce={"age": "int"}
))

## Demo 52 — Audit Tool Definitions

Read tool definitions from JSON (auto-detects OpenAI, Anthropic, MCP, Google, or JSON Schema format) and report per-tool token costs plus cross-format comparison.

In [ ]:
from agent_friend.audit import parse_tools, generate_report
import json

# Sample MCP server tools
mcp_tools = [
    {"name": "read_file", "description": "Read a file from the filesystem", "inputSchema": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}},
    {"name": "search", "description": "Search the web for information and return results", "inputSchema": {"type": "object", "properties": {"query": {"type": "string"}, "num_results": {"type": "integer"}}, "required": ["query"]}},
    {"name": "execute", "description": "Execute a shell command", "inputSchema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}}
]

tools = parse_tools(mcp_tools)
print(generate_report(tools, use_color=False))
